# 가중치 민감도 검증 — 30/30/20/20을 흔들면 판정이 얼마나 바뀌나

산식 가중치(주행거리 30 / 생활권 안 안전 30 / 밖 안전 20 / 패턴 안정성 20)를
바꿔가며 180개 시나리오의 **연간 우대 판정**을 다시 계산한다.

- 케어 판정은 가중치를 쓰지 않으므로(변화 지수 기반) 영향이 없다 — 우대 축만 재판정한다.
- 월별 축 점수는 시뮬레이션 번들에 저장된 값을 그대로 쓰고, 가중치만 바꾼다.
- 우대 규칙은 엔진과 동일: 통합점수 75점 이상, 평가 12개월 중 9개월 충족, 커버리지 80% 미만은 보류.

**주의**: 합성 시나리오 기반이므로 "이 값이 옳다"의 증거가 아니다.
값을 흔들었을 때 결과가 얼마나 움직이는지(둔감성)를 보는 용도다.

In [ ]:
# 데이터 로드 — 로컬 저장소가 있으면 그 파일을, 없으면(Colab) GitHub에서 받는다
import json, os, urllib.request
from pathlib import Path

LOCAL = Path("data/fixtures/gaip_simulation_bundle.json")
RAW_URL = ("https://raw.githubusercontent.com/summit1123/seniorcareservice/"
           "claude/gaip-dashboard-refine/data/fixtures/gaip_simulation_bundle.json")

if LOCAL.exists():
    bundle = json.loads(LOCAL.read_text(encoding="utf-8"))
elif Path("../data/fixtures/gaip_simulation_bundle.json").exists():
    bundle = json.loads(Path("../data/fixtures/gaip_simulation_bundle.json").read_text(encoding="utf-8"))
else:
    print("로컬 파일이 없어 GitHub에서 내려받습니다...")
    with urllib.request.urlopen(RAW_URL) as r:
        bundle = json.loads(r.read().decode("utf-8"))

drivers = bundle["drivers"]
print(f"시나리오 {len(drivers)}건 로드 완료")

In [ ]:
# 재판정 함수 — 엔진의 우대 규칙을 그대로 재현
AXES = ("mileage_score", "in_zone_safe_score", "out_zone_safe_score", "pattern_stability_score")
REWARD_THRESHOLD = 75.0
REWARD_REQUIRED_MONTHS = 9
MIN_COVERAGE_PCT = 80.0

def annual_reward_states(drivers, weights):
    w = dict(zip(AXES, weights))
    states = []
    for d in drivers:
        reward_months = eligible_months = 0
        for m in d["monthly_results"]:
            if m["period_role"] != "evaluation":
                continue
            if not m.get("zone_available") or float(m.get("data_coverage_pct", 0)) < MIN_COVERAGE_PCT:
                continue  # 보류 — 판정 미진입
            observed = [(float(m[a]), w[a]) for a in AXES if m.get(a) is not None]
            ow = sum(wt for _, wt in observed)
            if ow <= 0:
                continue
            eligible_months += 1
            score = sum(v * wt for v, wt in observed) / ow  # 미관측 축 재정규화
            if score >= REWARD_THRESHOLD:
                reward_months += 1
        if eligible_months < REWARD_REQUIRED_MONTHS:
            states.append("hold")
        elif reward_months >= REWARD_REQUIRED_MONTHS:
            states.append("reward")
        else:
            states.append("neutral")
    return states

In [ ]:
# 가중치 변형별 재판정
VARIANTS = {
    "30/30/20/20 (현행)": (30, 30, 20, 20),
    "30/30/15/25":        (30, 30, 15, 25),
    "30/30/25/15":        (30, 30, 25, 15),
    "25/35/20/20":        (25, 35, 20, 20),
    "25/25/25/25 (균등)": (25, 25, 25, 25),
    "35/25/20/20":        (35, 25, 20, 20),
    "20/40/20/20":        (20, 40, 20, 20),
    "40/30/15/15":        (40, 30, 15, 15),
}

base = annual_reward_states(drivers, VARIANTS["30/30/20/20 (현행)"])
print(f"현행: 우대 {base.count('reward')} / 중립 {base.count('neutral')} / 보류 {base.count('hold')}\n")

results = {}
print(f"{'가중치':22s} {'우대':>4s} {'현행 대비 변경':>10s}")
print("-" * 44)
for name, weights in VARIANTS.items():
    states = annual_reward_states(drivers, weights)
    changed = sum(1 for a, b in zip(base, states) if a != b)
    results[name] = changed
    print(f"{name:22s} {states.count('reward'):4d} {changed:9d}건")

In [ ]:
# 시각화 — 변형별 판정 변경 건수
import matplotlib.pyplot as plt
import matplotlib

for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(f.lower() in x.name.lower() for x in matplotlib.font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f
        break
plt.rcParams["axes.unicode_minus"] = False

names = [n for n in results if "현행" not in n]
vals = [results[n] for n in names]
colors = ["#0f766e" if v <= 6 else "#d97706" for v in vals]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(names[::-1], vals[::-1], color=colors[::-1])
ax.set_xlabel("현행 대비 판정 변경 (건 / 180)")
ax.set_title("가중치 민감도 — ±5 조정에서는 0~6건, 주행거리 40에서만 23건")
for b, v in zip(bars, vals[::-1]):
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height()/2, str(v), va="center")
ax.axvline(6.5, color="#999", ls="--", lw=0.8)
plt.tight_layout(); plt.show()

## 해석

- **±5 범위의 조정에서는 180건 중 0~6건만 바뀐다.** 밖 안전과 패턴 안정성을 맞바꾼
  30/30/15/25는 변경이 0건이다. 우리가 주장하는 것은 정확한 값이 아니라
  **순서**(노출·주 무대 ≥ 보조 무대·변화)라는 뜻이다.
- **주행거리 축이 가장 민감하다.** 40까지 올리면 23건이 바뀐다(우대 120→107).
  저주행 고객에게 유리한 축이라, 이 축을 키우면 활동적 안전 운전자가 탈락한다.
- **한계**: 합성 시나리오 기반이므로 "이 값이 옳다"의 증거가 아니다. 실측 손해율이
  쌓이면 그 데이터로 재보정한다(가중치는 전부 선언적 설정값이라 구조 변경 불필요).